# 06 — Model Building

Baseline linear regression → multi-model comparison across 7 algorithms.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import joblib, yaml, os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','axes.labelcolor':'#e2e8f0',
    'xtick.color':'#94a3b8','ytick.color':'#94a3b8','text.color':'#e2e8f0','grid.color':'#334155'})
PALETTE = ['#38bdf8','#fb7185','#34d399','#fbbf24','#a78bfa','#f97316','#e879f9','#22d3ee']

with open('../configs/paths.yaml') as f:
    paths = yaml.safe_load(f)
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
with open('../configs/model.yaml') as f:
    mcfg = yaml.safe_load(f)

os.makedirs(f'../{paths["artifacts"]["models"]}', exist_ok=True)
os.makedirs(f'../{paths["artifacts"]["outputs"]}', exist_ok=True)

In [ ]:
# ── Load engineered data ──────────────────────────────────────────────────────
df = pd.read_csv(f'../{paths["data"]["processed"]}data_engineered.csv')
TARGET = cfg['project']['target']
X = df.drop(columns=[TARGET])
y = df[TARGET]
print(f'X: {X.shape}  |  y: {y.shape}')
print('Features:', X.columns.tolist())

In [ ]:
# ── Time-based train/test split (last 20% as test) ────────────────────────────
RANDOM_STATE = cfg['project']['random_state']
TEST_SIZE = cfg['project']['test_size']

split_idx = int(len(X) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

# Scale numeric features
NUM_COLS = cfg['features']['numeric']
scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s  = X_test.copy()
X_train_s[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test_s[NUM_COLS]  = scaler.transform(X_test[NUM_COLS])

In [ ]:
# ── Helper: evaluate a model ──────────────────────────────────────────────────
def evaluate(name, model, Xtr, ytr, Xte, yte, needs_scaled=False):
    Xtrain = X_train_s if needs_scaled else Xtr
    Xtest  = X_test_s  if needs_scaled else Xte
    model.fit(Xtrain, ytr)
    pred   = model.predict(Xtest)
    mae    = mean_absolute_error(yte, pred)
    rmse   = np.sqrt(mean_squared_error(yte, pred))
    r2     = r2_score(yte, pred)
    cv_r2  = cross_val_score(model, Xtrain, ytr, cv=5, scoring='r2').mean()
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
            'CV_R2': cv_r2, 'Predictions': pred}

In [ ]:
# ── Train all models ──────────────────────────────────────────────────────────
results = []

models = [
    ('Linear Regression',    LinearRegression(),                               True),
    ('Ridge',                Ridge(alpha=mcfg['ridge']['alpha']),               True),
    ('Lasso',                Lasso(alpha=mcfg['lasso']['alpha']),               True),
    ('Decision Tree',        DecisionTreeRegressor(**{k:v for k,v in mcfg['decision_tree'].items()}), False),
    ('Random Forest',        RandomForestRegressor(**{k:v for k,v in mcfg['random_forest'].items()}), False),
    ('Gradient Boosting',    GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42), False),
    ('XGBoost',              xgb.XGBRegressor(**{k:v for k,v in mcfg['xgboost'].items()}, verbosity=0), False),
    ('LightGBM',             lgb.LGBMRegressor(**{k:v for k,v in mcfg['lightgbm'].items()}, verbose=-1), False),
    ('CatBoost',             CatBoostRegressor(**{k:v for k,v in mcfg['catboost'].items()}), False),
]

all_preds = {}
for name, model, scaled in models:
    r = evaluate(name, model, X_train, y_train, X_test, y_test, needs_scaled=scaled)
    all_preds[name] = r.pop('Predictions')
    results.append(r)
    print(f'{name:<22}  R2={r["R2"]:.4f}  RMSE={r["RMSE"]:.1f}  MAE={r["MAE"]:.1f}  CV_R2={r["CV_R2"]:.4f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# ── Model comparison bar chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Comparison on Test Set', fontsize=14)

for ax, metric, title in zip(axes, ['R2','RMSE','MAE'], ['R² Score','RMSE','MAE']):
    data = results_df.sort_values(metric, ascending=(metric!='R2'))
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(data))]
    ax.barh(data['Model'], data[metric], color=colors, edgecolor='#0f172a')
    ax.set_title(title)
    ax.set_xlabel(metric)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Actual vs Predicted for top 3 models ─────────────────────────────────────
top3 = results_df.head(3)['Model'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Actual vs Predicted — Top 3 Models')

for ax, name in zip(axes, top3):
    ax.scatter(y_test, all_preds[name], alpha=0.5, color=PALETTE[0], s=25)
    lo = min(y_test.min(), all_preds[name].min())
    hi = max(y_test.max(), all_preds[name].max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Perfect')
    r2 = results_df.loc[results_df['Model']==name,'R2'].values[0]
    ax.set(title=f'{name}\nR²={r2:.4f}', xlabel='Actual', ylabel='Predicted')
    ax.legend(); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../outputs/actual_vs_predicted_top3.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Identify and save champion model ─────────────────────────────────────────
champion_name = results_df.iloc[0]['Model']
print(f'Champion: {champion_name}')

# Re-train champion on full training set with correct settings
champion_model_map = {
    'Random Forest':     RandomForestRegressor(**{k:v for k,v in mcfg['random_forest'].items()}),
    'XGBoost':           xgb.XGBRegressor(**{k:v for k,v in mcfg['xgboost'].items()}, verbosity=0),
    'LightGBM':          lgb.LGBMRegressor(**{k:v for k,v in mcfg['lightgbm'].items()}, verbose=-1),
    'CatBoost':          CatBoostRegressor(**{k:v for k,v in mcfg['catboost'].items()}),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    'Decision Tree':     DecisionTreeRegressor(**{k:v for k,v in mcfg['decision_tree'].items()}),
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=mcfg['ridge']['alpha']),
    'Lasso':             Lasso(alpha=mcfg['lasso']['alpha']),
}
champion = champion_model_map[champion_name]
champion.fit(X_train, y_train)
joblib.dump(champion, f'../{paths["artifacts"]["models"]}champion_model.pkl')
print('Champion model saved.')

## Model Building Summary

| Step | Detail |
|---|---|
| Split strategy | Time-based (last 20% as test) |
| CV strategy | 5-fold on training set |
| Models compared | 9 algorithms |
| Champion selection | Highest test R² |

**Next:** `07_Hyperparameter_Tuning.ipynb`
